# Cybersecurity Incident Analyzer — Fase 1

## Análise de Incidentes de Segurança com Modelos de NLP Pré-Treinados

---

### Objetivo do Projeto

O **Cybersecurity Incident Analyzer** é um sistema que auxilia equipes de **SOC (Security Operations Center)** na triagem e análise de incidentes de segurança da informação. O projeto é dividido em fases incrementais, cada uma adicionando uma camada de complexidade e capacidade técnica.

### O Problema

Equipes de SOC recebem diariamente um grande volume de alertas e relatos de incidentes — phishing, malware, tentativas de acesso não autorizado, exfiltração de dados, entre outros. A triagem manual desses incidentes é:

- **Lenta**: analistas precisam ler relatórios extensos antes de agir;
- **Sujeita a erros**: classificação inconsistente entre analistas diferentes;
- **Difícil de escalar**: o volume de alertas cresce mais rápido que o quadro de analistas.

Técnicas de **Processamento de Linguagem Natural (NLP)** podem auxiliar automatizando partes desse fluxo: resumir relatos longos, classificar o tipo de incidente automaticamente e sugerir ações iniciais de resposta.

### Objetivo da Fase 1

Esta fase tem foco **exclusivamente** na aplicação de modelos de NLP pré-treinados (Hugging Face) como componentes técnicos isolados, demonstrando:

1. **Sumarização** de relatos de incidentes;
2. **Classificação Zero-Shot** da categoria do incidente;
3. **Geração de recomendações** de resposta inicial;
4. Conceitos fundamentais de **tokenização** e **arquiteturas de modelos de linguagem**.

> ⚠️ **Fora de escopo nesta fase:** RAG, embeddings, bancos de dados vetoriais e recuperação documental. Esses componentes serão introduzidos em fases futuras, quando o sistema passará a fundamentar suas respostas em uma base de conhecimento externa (NIST, OWASP, MITRE ATT&CK, CISA, CVE).

### Ambiente recomendado

Este notebook foi projetado para rodar no **Google Colab** com runtime de **GPU (T4 ou superior)**. Para ativar: `Ambiente de execução > Alterar o tipo de ambiente de execução > GPU`.


---
## 1. Instalação de Dependências e Imports

Instalamos as bibliotecas necessárias: `transformers` (modelos e pipelines da Hugging Face), `torch` (backend de deep learning), `pandas` e `numpy` (manipulação de dados).


In [ ]:
# Instalação das dependências (descomente se necessário no Colab)
!pip install -q transformers torch pandas numpy sentencepiece accelerate


In [ ]:
import torch
import pandas as pd
import numpy as np
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
)

# Verifica se há GPU disponível
device = 0 if torch.cuda.is_available() else -1
device_name = "GPU" if device == 0 else "CPU"
print(f"Dispositivo em uso: {device_name}")

if torch.cuda.is_available():
    print(f"GPU detectada: {torch.cuda.get_device_name(0)}")


---
## 2. Dataset de Incidentes (Exemplos Fictícios)

Criamos um pequeno conjunto de incidentes fictícios cobrindo diferentes categorias comuns em ambientes corporativos. Esses dados serão usados para demonstrar as três tarefas de NLP ao longo do notebook.

Devido aos modelos serem treinados em inglês, os relatos dos incidentes estão em inglês, mas as explicações e comentários estão em português para facilitar a compreensão.

> Nota: todos os dados abaixo são **fictícios**, criados exclusivamente para fins didáticos.


In [ ]:
incidents = [
    {
        "id": 1,
        "text": (
            "On June 14th, multiple employees in the Finance department reported receiving "
            "an email claiming to be from the CFO, requesting urgent wire transfers to a new "
            "vendor account. The email contained a spoofed domain (cfo-finance-corp.com) and "
            "urged recipients to act within one hour to avoid 'contract penalties'. One employee "
            "clicked the link and entered their corporate credentials on a fake Office 365 login "
            "page before realizing the site was illegitimate. IT Security was notified and the "
            "employee's account was immediately disabled pending investigation."
        ),
        "expected_category": "Phishing",
    },
    {
        "id": 2,
        "text": (
            "At 03:12 AM, the file server hosting the HR department's shared drive became "
            "unresponsive. Upon investigation, system administrators discovered that thousands "
            "of files had been encrypted and renamed with a '.lockedcorp' extension. A ransom "
            "note demanding 5 Bitcoin was found in every affected directory, threatening to leak "
            "employee personal data if payment was not made within 72 hours. Backups from the "
            "previous night appear unaffected and are being verified for integrity."
        ),
        "expected_category": "Ransomware",
    },
    {
        "id": 3,
        "text": (
            "Endpoint detection software flagged unusual behavior on a marketing department "
            "laptop: a previously unseen executable was making repeated outbound connections to "
            "an IP address associated with known command-and-control infrastructure. The process "
            "was attempting to enumerate browser-stored credentials and capture keystrokes. The "
            "laptop was isolated from the network and a forensic image was taken for analysis."
        ),
        "expected_category": "Malware",
    },
    {
        "id": 4,
        "text": (
            "Security monitoring detected that an employee's VPN credentials were used to log in "
            "simultaneously from two geographically distant locations — one from the corporate "
            "office in Sao Paulo and another from an IP address registered in Eastern Europe, "
            "less than ten minutes apart. The employee confirmed they had not traveled and had "
            "not shared their password. A password reset and credential rotation was initiated "
            "immediately, along with a review of recently accessed systems."
        ),
        "expected_category": "Credential Theft",
    },
    {
        "id": 5,
        "text": (
            "A system administrator with privileged access to the customer database was found to "
            "have exported a large volume of customer records to a personal USB drive shortly "
            "after receiving a negative performance review. The export occurred outside of normal "
            "working hours and was not associated with any approved business task. Data Loss "
            "Prevention (DLP) tooling flagged the transfer for review."
        ),
        "expected_category": "Insider Threat",
    },
    {
        "id": 6,
        "text": (
            "The authentication logs for the company's customer-facing web portal show over "
            "40,000 failed login attempts within a two-hour window, originating from a distributed "
            "set of IP addresses and targeting a list of common username patterns. The pattern is "
            "consistent with an automated credential-stuffing or brute-force attack attempting to "
            "compromise customer accounts using leaked password lists."
        ),
        "expected_category": "Brute Force Attack",
    },
    {
        "id": 7,
        "text": (
            "The company's primary e-commerce website became completely unreachable for "
            "approximately 45 minutes during a major promotional sale. Network monitoring showed "
            "an abnormal surge of incoming traffic, estimated at over 200 times normal volume, "
            "originating from thousands of distinct IP addresses worldwide and overwhelming the "
            "load balancers. The traffic pattern strongly suggests a coordinated denial-of-service "
            "attack timed to coincide with peak sales traffic."
        ),
        "expected_category": "DDoS",
    },
    {
        "id": 8,
        "text": (
            "A help desk technician received a phone call from someone claiming to be a senior "
            "executive who had 'forgotten' their password and urgently needed it reset to access "
            "an important client presentation. The caller was able to provide the executive's "
            "employee ID and date of birth, likely gathered from a previous data breach, and "
            "pressured the technician to bypass standard identity verification procedures."
        ),
        "expected_category": "Phishing",
    },
    {
        "id": 9,
        "text": (
            "Anti-virus software on a developer's workstation quarantined a file disguised as a "
            "legitimate npm package dependency that had been recently added to a project. Analysis "
            "revealed the package contained obfuscated code designed to harvest environment "
            "variables, including API keys and cloud credentials, and transmit them to an external "
            "server upon installation."
        ),
        "expected_category": "Malware",
    },
    {
        "id": 10,
        "text": (
            "Several customer support agents reported that their session tokens appeared to be "
            "reused from unfamiliar devices shortly after they had logged into the support "
            "ticketing platform from a coffee shop's public Wi-Fi network. Investigation suggests "
            "the sessions may have been intercepted via an unsecured network, allowing an attacker "
            "to hijack active sessions without needing the original password."
        ),
        "expected_category": "Credential Theft",
    },
]

df_incidents = pd.DataFrame(incidents)
print(f"Total de incidentes no dataset: {len(df_incidents)}")
df_incidents[["id", "expected_category"]]


---
## 3. Tarefa 1 — Sumarização de Incidentes

Relatos de incidentes em ambientes reais frequentemente vêm em formato de texto longo (tickets, e-mails, logs narrativos). A sumarização automática ajuda analistas a entender rapidamente o núcleo do incidente sem ler o relato completo.

Utilizamos o modelo **`facebook/bart-large-cnn`**, um modelo encoder-decoder pré-treinado especificamente para tarefas de sumarização de texto em inglês.


In [ ]:
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=device,
)


In [ ]:
# Demonstração com o incidente de ransomware (id=2), que possui um relato mais longo
sample_incident = df_incidents[df_incidents["id"] == 2].iloc[0]


print("TEXTO ORIGINAL:")

print(sample_incident["text"])
print()

summary = summarizer(
    sample_incident["text"],
    max_length=60,
    min_length=15,
    do_sample=False,
)[0]["summary_text"]


print("RESUMO GERADO:")

print(summary)


In [ ]:
# Aplicando sumarização a todos os incidentes do dataset
summaries = []
for incident in incidents:
    result = summarizer(
        incident["text"],
        max_length=50,
        min_length=10,
        do_sample=False,
    )[0]["summary_text"]
    summaries.append(result)

df_incidents["summary"] = summaries
df_incidents[["id", "expected_category", "summary"]]


### Observações sobre a qualidade dos resumos

- O modelo `bart-large-cnn` foi treinado majoritariamente em artigos de notícias (dataset CNN/DailyMail), então tende a produzir resumos com tom jornalístico, o que funciona razoavelmente bem para relatos narrativos de incidentes.
- Em textos curtos, o resumo pode ficar muito próximo do texto original — isso é esperado, já que há pouco a comprimir.
- Para textos técnicos com muita terminologia específica (nomes de malware, CVEs, hashes), o modelo pode omitir detalhes tecnicamente relevantes por não reconhecer sua importância — uma limitação que reforça a necessidade de revisão humana ou, futuramente, de um modelo ajustado ao domínio de segurança.


---
## 4. Tarefa 2 — Classificação Zero-Shot de Incidentes

Classificação **zero-shot** permite categorizar textos em rótulos definidos **sem necessidade de treinamento específico** para essas categorias. O modelo avalia o quão bem cada rótulo "se encaixa" como hipótese para o texto, usando uma abordagem de **Natural Language Inference (NLI)**.

Utilizamos **`facebook/bart-large-mnli`**, treinado no dataset MultiNLI.


In [ ]:
zero_shot_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device,
)

candidate_labels = [
    "Phishing",
    "Malware",
    "Ransomware",
    "Credential Theft",
    "DDoS",
    "Insider Threat",
    "Brute Force Attack",
]


In [ ]:
def classify_incident(text, labels=candidate_labels):
    result = zero_shot_classifier(text, candidate_labels=labels, multi_label=False)
    predicted_label = result["labels"][0]
    confidence = result["scores"][0]
    return predicted_label, confidence


# Classificando todos os incidentes do dataset
predicted_categories = []
confidence_scores = []

for incident in incidents:
    label, score = classify_incident(incident["text"])
    predicted_categories.append(label)
    confidence_scores.append(round(score, 4))

df_incidents["predicted_category"] = predicted_categories
df_incidents["confidence"] = confidence_scores

df_incidents[["id", "expected_category", "predicted_category", "confidence"]]


In [ ]:
# Avaliação simples: quantos incidentes foram classificados corretamente?
correct = (df_incidents["expected_category"] == df_incidents["predicted_category"]).sum()
total = len(df_incidents)
accuracy = correct / total

print(f"Classificações corretas: {correct}/{total} ({accuracy:.0%})")
print()
print("Incidentes classificados incorretamente:")
mismatches = df_incidents[df_incidents["expected_category"] != df_incidents["predicted_category"]]
if mismatches.empty:
    print("Nenhum.")
else:
    print(mismatches[["id", "expected_category", "predicted_category", "confidence"]])


### Observações

- A classificação zero-shot é poderosa por não exigir dados de treinamento rotulados para o domínio específico, mas sua precisão depende de quão bem os **nomes dos rótulos** descrevem semanticamente a categoria.
- Categorias com vocabulário sobreposto (ex: *Credential Theft* vs. *Phishing*, já que phishing é frequentemente o vetor usado para roubo de credenciais) tendem a gerar confusão e scores de confiança mais baixos.

//TODO avaliar se precisa mudar para multi_label=True
- O parâmetro `multi_label=False` força uma única categoria predominante; em cenários reais, um incidente pode pertencer a múltiplas categorias simultaneamente (`multi_label=True` seria mais apropriado nesse caso).


---
## 5. Tarefa 3 — Geração de Recomendações de Resposta

Nesta etapa, utilizamos um modelo generativo para produzir uma análise estruturada do incidente: categoria, severidade e ações recomendadas.

> **Nota sobre escolha de modelo:** a especificação original sugeria `microsoft/Phi-3-mini-4k-instruct`. Optamos por **`google/flan-t5-large`** como alternativa mais leve — ele é um modelo encoder-decoder ajustado por instruções (*instruction-tuned*), com bom desempenho em tarefas estruturadas e custo computacional bem menor, o que o torna mais adequado para ambientes com GPU limitada (como o T4 padrão do Colab gratuito). Se você tiver acesso a uma GPU mais robusta, basta trocar o nome do modelo na célula abaixo por `microsoft/Phi-3-mini-4k-instruct` (ajustando o pipeline para `text-generation`).


In [ ]:
GENERATION_MODEL = "google/flan-t5-large"  # Alternativa leve ao Phi-3-mini

gen_tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(GENERATION_MODEL)

if torch.cuda.is_available():
    gen_model = gen_model.to("cuda")

print(f"Modelo de geração carregado: {GENERATION_MODEL}")


In [ ]:
def build_prompt(incident_text):
    return (
        "You are a cybersecurity analyst. Analyze the following security incident report "
        "and respond with exactly three labeled sections.\n\n"
        f"Incident report:\n{incident_text}\n\n"
        "1. Incident category:\n"
        "2. Severity (Low, Medium, High, Critical):\n"
        "3. Recommended actions:"
    )


def generate_recommendation(incident_text, temperature=0.3, max_new_tokens=150):
    prompt = build_prompt(incident_text)
    inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)

    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    do_sample = temperature > 0.0
    output_ids = gen_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        top_p=0.9 if do_sample else None,
    )
    return gen_tokenizer.decode(output_ids[0], skip_special_tokens=True)


In [ ]:
# Demonstrando para 3 incidentes representativos: phishing, ransomware e insider threat
demo_ids = [1, 2, 5]

for incident_id in demo_ids:
    incident = next(i for i in incidents if i["id"] == incident_id)
    
    print(f"INCIDENTE #{incident['id']} (categoria esperada: {incident['expected_category']})")
    
    print(incident["text"])
    print()
    recommendation = generate_recommendation(incident["text"], temperature=0.3)

    print("ANÁLISE GERADA PELO MODELO:")
    print("-" * 80)
    print(recommendation)


### Observações sobre a Tarefa 3

- Modelos generativos de porte menor como `flan-t5-large` tendem a produzir respostas mais curtas e diretas — adequado para um primeiro protótipo, mas com profundidade técnica limitada quando comparado a modelos maiores como Phi-3 ou GPT-4-class.
- A qualidade das recomendações depende fortemente do **prompt** fornecido; pequenas mudanças na estrutura do prompt podem mudar significativamente a saída.
- Sem acesso a uma base de conhecimento (frameworks como NIST SP 800-61 ou MITRE ATT&CK), as recomendações refletem apenas o conhecimento genérico absorvido durante o pré-treinamento — esse é exatamente o gap que a futura fase de RAG pretende resolver.


---
## 6. Tokenização

Antes de qualquer modelo processar texto, é necessário convertê-lo em **tokens** — unidades discretas (palavras, subpalavras ou caracteres) que o modelo entende numericamente. Cada modelo possui seu próprio tokenizador, treinado junto com o vocabulário do modelo.

A escolha do tokenizador afeta diretamente:
- o **tamanho máximo de contexto** que pode ser processado (limite de tokens, não de caracteres);
- o **custo computacional** (mais tokens = mais processamento);
- a **qualidade da representação** de termos técnicos ou fora do vocabulário comum (ex: nomes de malware, hashes, CVEs).


In [ ]:
demo_tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")

sample_text = incidents[1]["text"]  # incidente de ransomware

tokens = demo_tokenizer.tokenize(sample_text)
token_ids = demo_tokenizer.encode(sample_text)

print("TEXTO ORIGINAL (primeiros 200 caracteres):")
print(sample_text[:200] + "...")
print()
print(f"Número de caracteres: {len(sample_text)}")
print(f"Número de tokens: {len(tokens)}")
print()
print("Primeiros 30 tokens gerados:")
print(tokens[:30])
print()
print("Primeiros 30 IDs correspondentes:")
print(token_ids[:30])


### Por que a tokenização importa

- O exemplo acima mostra que o número de tokens **não é igual** ao número de palavras nem ao número de caracteres — modelos baseados em **subword tokenization** (como BPE, usado pelo BART) podem dividir uma palavra em múltiplos tokens, especialmente termos raros ou técnicos.
- Isso é particularmente relevante em segurança da informação: termos como nomes de CVEs, hashes (`SHA-256`), endereços IP ou nomes de malware podem ser fragmentados de formas que dificultam o entendimento do modelo sobre o seu significado.
- Todo modelo possui um **limite máximo de tokens de entrada** (ex: 1024 para BART, 512 para muitos modelos T5). Textos de incidentes muito longos precisam ser truncados ou divididos em chunks — um problema que será mais relevante ainda nas fases futuras com RAG.


---
## 7. Comparação de Arquiteturas de Modelos de Linguagem

Os modelos utilizados neste notebook pertencem a três famílias arquiteturais distintas. Entender essas diferenças ajuda a escolher o modelo certo para cada tarefa.

### Encoder-only

**Exemplos:** BERT, DistilBERT, RoBERTa

**Como funciona:** processa o texto de entrada bidirecionalmente (cada token "vê" todos os outros tokens da sequência, antes e depois dele) e produz representações vetoriais (embeddings) contextuais.

**Usos típicos:**
- Classificação de texto (ex: a base por trás de `bart-large-mnli` no componente de classificação)
- Geração de embeddings para busca semântica
- Named Entity Recognition (NER) — útil para extrair IOCs (Indicators of Compromise) de relatos de incidentes

### Decoder-only

**Exemplos:** GPT (família), Llama, Phi-3, Mistral

**Como funciona:** processa o texto de forma autoregressiva e unidirecional (da esquerda para a direita), prevendo o próximo token com base apenas nos tokens anteriores.

**Usos típicos:**
- Geração de texto livre
- Chatbots e assistentes conversacionais
- Tarefas que exigem raciocínio sequencial e geração extensa, como a análise detalhada de incidentes (Tarefa 3 deste notebook, em sua versão original com Phi-3)

### Encoder-Decoder

**Exemplos:** BART, T5, FLAN-T5

**Como funciona:** combina um encoder (que lê e compreende a entrada) com um decoder (que gera a saída), conectados por um mecanismo de atenção cruzada. É a arquitetura clássica para tarefas "sequência-para-sequência".

**Usos típicos:**
- Sumarização (Tarefa 1 — `bart-large-cnn`)
- Tradução automática
- Tarefas estruturadas de instrução-resposta (Tarefa 3 deste notebook, com `flan-t5-large`)

### Resumo comparativo

| Arquitetura | Direção do contexto | Pontos fortes | Usado neste notebook para |
|---|---|---|---|
| Encoder-only | Bidirecional | Compreensão, classificação | Classificação zero-shot |
| Decoder-only | Unidirecional (esquerda→direita) | Geração livre, chat | (alternativa: Phi-3) |
| Encoder-Decoder | Bidirecional na entrada, unidirecional na saída | Transformação estruturada de texto | Sumarização e geração de recomendações |


---
## 8. Experimentos com Parâmetros de Geração

Modelos generativos possuem parâmetros que controlam o **comportamento da amostragem** durante a geração de texto. Os dois mais importantes são:

- **`temperature`**: controla a aleatoriedade da escolha do próximo token. Valores próximos de 0 tornam a saída mais determinística (o modelo sempre escolhe os tokens mais prováveis); valores mais altos (ex: 0.8–1.2) aumentam a diversidade e criatividade, mas também o risco de respostas incoerentes ou alucinadas.
- **`max_new_tokens`**: limite máximo de tokens que o modelo pode gerar na resposta. Afeta diretamente o nível de detalhe da saída e o tempo de processamento.

Vamos comparar como esses parâmetros afetam a análise gerada para o mesmo incidente.


In [ ]:
experiment_incident = next(i for i in incidents if i["id"] == 6)  # Brute Force Attack

print("INCIDENTE USADO NO EXPERIMENTO:")
print(experiment_incident["text"])
print()

temperature_settings = [0.0, 0.7, 1.2]

for temp in temperature_settings:
    
    print(f"TEMPERATURE = {temp}")
    
    result = generate_recommendation(experiment_incident["text"], temperature=temp, max_new_tokens=120)
    print(result)
    print()


In [ ]:
# Comparando diferentes valores de max_new_tokens (com temperature fixa)
token_limits = [30, 80, 150]

for max_tokens in token_limits:
    
    print(f"MAX_NEW_TOKENS = {max_tokens}")
    
    result = generate_recommendation(experiment_incident["text"], temperature=0.3, max_new_tokens=max_tokens)
    print(result)
    print()


### Relacionando os parâmetros ao contexto de segurança

- **Em produção, para análise de incidentes reais, recomenda-se `temperature` baixa (próxima de 0)**: a consistência e a previsibilidade são mais valiosas do que a criatividade quando se trata de orientar decisões de resposta a incidentes. Uma recomendação "criativa" pode significar uma recomendação **incorreta** ou inadequada ao contexto real.
- `max_new_tokens` muito baixo pode truncar uma recomendação importante no meio (por exemplo, cortar a lista de ações recomendadas antes do item mais crítico); muito alto pode gerar texto repetitivo ou divagante, especialmente em modelos menores como o usado aqui.

---
## 9. Limitações Observadas Nesta Fase

Ao longo dos experimentos deste notebook, diversas limitações ficaram evidentes:

### Alucinações
Modelos generativos podem produzir recomendações que **parecem plausíveis mas não correspondem a práticas reais de resposta a incidentes**, especialmente quando o prompt não fornece contexto suficiente ou quando o tema é muito específico do domínio.

### Classificações incorretas
Na Tarefa 2, observamos que categorias com sobreposição semântica (como *Phishing* e *Credential Theft*) podem ser confundidas pelo classificador zero-shot, já que ambas compartilham vocabulário relacionado a credenciais e engenharia social.

### Dependência do conhecimento pré-treinado
Todos os modelos utilizados aqui — sumarização, classificação e geração — baseiam suas respostas **exclusivamente no que aprenderam durante o pré-treinamento**. Eles não têm acesso a:
- frameworks de segurança específicos da organização;
- playbooks de resposta a incidentes customizados;
- histórico de incidentes anteriores da própria empresa;
- informações atualizadas sobre ameaças emergentes (zero-days recentes, CVEs publicados após o corte de treinamento do modelo).

### Ausência de conhecimento específico da organização
Um SOC real opera sob políticas, ferramentas e processos próprios. Um modelo genérico não conhece, por exemplo, qual é o procedimento de escalonamento específico da empresa, ou quais sistemas são considerados críticos no ambiente daquela organização.

### Conclusão

Essas limitações apontam para a necessidade de uma fase futura que incorpore **Retrieval-Augmented Generation (RAG)** — uma arquitetura em que o modelo generativo é alimentado com trechos relevantes de documentos externos (frameworks de segurança, playbooks internos, bases de CVE) recuperados dinamicamente a partir da pergunta ou do incidente analisado. Isso permitirá que as respostas do sistema sejam **fundamentadas (grounded)** em fontes confiáveis e atualizáveis, em vez de dependerem unicamente do conhecimento estático adquirido durante o pré-treinamento do modelo.

Essa será o foco da **Fase 2** deste projeto. Os materiais recomendados para compor a base de conhecimento dessa próxima fase estão documentados em `docs/recommended_corpus.md`.


---
## 10. Conclusão da Fase 1

Neste notebook, demonstramos a aplicação de três técnicas centrais de NLP — sumarização, classificação zero-shot e geração de texto guiada por instrução — a um conjunto de incidentes de segurança fictícios, cobrindo também conceitos fundamentais de tokenização e arquiteturas de modelos de linguagem.

Os resultados mostram potencial claro para auxiliar a triagem inicial de incidentes, mas também revelam limitações importantes relacionadas à ausência de conhecimento específico de domínio e de contexto organizacional.
